# 01 — Target Surface Sanity

## What this notebook does
Loads a target protein structure from the RCSB database, inspects its chains and residue
numbering, lets you define a binding pocket, and exports a frozen **target spec** that all
subsequent notebooks will load and trust.

## What decision it helps make
> "I have confirmed the correct chain, residue numbering, and pocket definition.
> I am ready to start scoring peptides."

## What it cannot prove
- That the pocket you defined is biologically correct
- That the PDB structure is the right conformation for your design goal
- Anything about peptide binding or affinity

---

> **Why this notebook exists:**
> The most common source of invalid results in peptide modeling is not bad scoring code.
> It is loading the wrong chain, using the wrong residue numbers, or never checking
> which part of the structure is the actual target surface.
> This notebook forces you to make those decisions explicitly and write them down
> before any scoring runs.

## Free vs Paid Colab

This notebook runs fully on **free Colab**. It has no GPU requirement.
It only downloads a PDB file (< 1 MB) and runs lightweight Python.

| Step | Free | Paid |
|------|------|------|
| Fetch PDB | Yes | Yes |
| Parse chains | Yes | Yes |
| Validate pocket | Yes | Yes |
| Export target spec | Yes | Yes |

In [ ]:
# Install required packages
# biopython: for PDB parsing
# requests is already available in Colab
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "biopython"])
print("Dependencies ready.")

In [ ]:
import sys, pathlib

# ── Environment detection ─────────────────────────────────────────────────
# Handles both local Jupyter (CWD = notebooks/) and Google Colab (CWD = /content/).
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not pathlib.Path('/content/cookbooks').exists():
        import subprocess
        subprocess.run(
            ['git', 'clone', '--depth=1',
             'https://github.com/peptidemodel/cookbooks', '/content/cookbooks'],
            check=True,
        )
    COOKBOOK_DIR = pathlib.Path('/content/cookbooks/colab-basics')
    # ── Optional: persist outputs to Google Drive ──────────────────────────
    # Uncomment the two lines below to save to Drive instead of session-only /content/:
    # from google.colab import drive; drive.mount('/content/drive')
    # WORKSPACE_DIR = pathlib.Path('/content/drive/MyDrive/peptide_workspace')
    WORKSPACE_DIR = pathlib.Path('/content/workspace')
else:
    COOKBOOK_DIR = pathlib.Path('..').resolve()
    WORKSPACE_DIR = COOKBOOK_DIR / 'workspace'

# ============================================================
# CONFIGURATION — edit the values below for your target
# ============================================================

# Compute tier — controls panel/scan size caps only.
# No GPU lane is active in this cookbook; all scoring runs on CPU.
# Set "paid" to raise the variant cap in notebook 04.
COMPUTE_TIER = "free"  # "free" or "paid"

# PDB ID of your target structure (4-letter code, from https://www.rcsb.org)
PDB_ID = "3HH2"  # teaching example — replace with your target

# Chain ID of the receptor/target chain (confirm below after inspecting the chain list)
TARGET_CHAIN_ID = "C"

# Pocket residue IDs as they appear in the PDB file.
# If unknown: set to a wide range (e.g. list(range(40, 120))), inspect below, then narrow.
POCKET_RESIDUE_IDS = [43, 44, 45, 78, 79, 80, 81, 82, 110, 111, 112]

POCKET_DESCRIPTION = "Activin type-II receptor binding surface (replace with your description)"
MECHANISM_HYPOTHESIS = (
    "Short peptides derived from activin type-II receptor interface "
    "may compete with the natural receptor binding surface."
)

# Positive reference peptide — use a real benchmark for your target.
POSITIVE_REFERENCE = "FSRIEGQYTKLNRSFM"  # example only — replace
POSITIVE_REFERENCE_SOURCE = "internal-proxy (replace with actual source)"

# Pocket property hints — used by the heuristic scoring lane (notebooks 02–04).
#
# pocket_net_charge_hint: net charge of the pocket surface.
#   Positive = cationic surface. Peptides with high charge magnitude score better.
#
# pocket_hydrophobic_hint: 0.0 = entirely polar/charged, 1.0 = entirely hydrophobic.
#   Set this to your pocket's actual character. This is the most important hint to get right.
#   COMMON MISTAKE: setting 0.55 for a polar interface. Ala hydrophobic fraction is ~0.62,
#   so poly-Ala would beat a polar reference on the hydrophobic sub-score.
#   If your panel returns DEGRADED or FAIL, review this value first.
POCKET_NET_CHARGE_HINT  = 2.0   # myostatin ActRII: charged interface
POCKET_HYDROPHOBIC_HINT = 0.30  # myostatin ActRII: predominantly polar/charged
OPTIMAL_PEPTIDE_LENGTH_MIN = 8
OPTIMAL_PEPTIDE_LENGTH_MAX = 20

print(f"Environment : {'Colab' if IN_COLAB else 'local Jupyter'}")
print(f"Cookbook dir: {COOKBOOK_DIR}")
print(f"Workspace   : {WORKSPACE_DIR}")
print(f"PDB: {PDB_ID}, Chain: {TARGET_CHAIN_ID}")

In [ ]:
# Setup workspace and shared library path
import sys, pathlib, json

WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

if str(COOKBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(COOKBOOK_DIR))

from shared.target_utils import (
    fetch_pdb, parse_pdb_chains, extract_pocket_residues,
    make_target_spec, save_target_spec, format_target_summary
)

print("Workspace ready. Shared utilities loaded.")

## Step 1 — Load the PDB Structure

We download the PDB file from the RCSB database. The PDB (Protein Data Bank) is a
public archive of 3D protein structures determined by X-ray crystallography, cryo-EM,
or NMR. Each structure has a 4-letter identifier.

If you have a local PDB file instead, load it with:
```python
pdb_text = pathlib.Path("my_structure.pdb").read_text()
```

In [ ]:
pdb_save_path = WORKSPACE_DIR / f"{PDB_ID}.pdb"
print(f"Fetching {PDB_ID} from RCSB...")
pdb_text = fetch_pdb(PDB_ID, save_path=pdb_save_path)
print(f"Downloaded {len(pdb_text)} characters. Saved to {pdb_save_path}")

## Step 2 — Inspect Chains

A PDB file typically contains multiple protein chains, ligands, water molecules,
and sometimes co-crystallised peptides. Before designing anything, you need to know:

- Which chain is the actual receptor/target?
- Does that chain have the residue numbering you expect?
- Are there any short chains that might be co-crystallised ligands or binding partners?

Check the chain list below and confirm `TARGET_CHAIN_ID` is what you intend.

In [ ]:
chain_info = parse_pdb_chains(pdb_text, structure_id=PDB_ID)

print(f"Structure: {chain_info['structure_id']}")
print(f"Total chains: {chain_info['total_chains']}")
print(f"Total residues: {chain_info['total_residues']}\n")

print(f"{'Chain':<8} {'AA Residues':<14} {'Range':<15} {'Note'}")
print("-" * 65)
for c in chain_info['chains']:
    rng = f"{c['residue_range'][0]}–{c['residue_range'][1]}"
    print(f"{c['chain_id']:<8} {c['n_amino_acid_residues']:<14} {rng:<15} {c['note']}")

print()
print(f">>> Your configured chain: {TARGET_CHAIN_ID}")
found_chain = any(c['chain_id'] == TARGET_CHAIN_ID for c in chain_info['chains'])
if found_chain:
    target_chain_data = next(c for c in chain_info['chains'] if c['chain_id'] == TARGET_CHAIN_ID)
    print(f"    Chain {TARGET_CHAIN_ID} found: {target_chain_data['n_amino_acid_residues']} "
          f"amino acid residues, range {target_chain_data['residue_range'][0]}–"
          f"{target_chain_data['residue_range'][1]}")
else:
    print(f"!!! Chain '{TARGET_CHAIN_ID}' NOT FOUND. Update TARGET_CHAIN_ID in the config cell.")

## Step 3 — Define and Validate the Binding Pocket

A **pocket** is the region of the protein surface where your peptide is intended to bind.
It is defined here as a list of residue numbers from the target chain.

Typical ways to identify pocket residues:
- Literature: the paper describing the target may list interface residues
- PyMOL/ChimeraX: visualise the structure and note residues at the interface
- fpocket / SiteMap: automated pocket detection tools

**Important:** The residue numbers here must match the PDB file's numbering,
not any renumbered or processed version. If you see numbering mismatches below,
fix `POCKET_RESIDUE_IDS` in the config cell before proceeding.

If you do not yet know the pocket residues:
- Set `POCKET_RESIDUE_IDS` to a range like `list(range(40, 90))` to inspect
  a segment, then narrow it down once you understand the structure.

In [ ]:
pocket_info = extract_pocket_residues(
    pdb_text=pdb_text,
    chain_id=TARGET_CHAIN_ID,
    pocket_residue_ids=POCKET_RESIDUE_IDS,
    structure_id=PDB_ID,
)

print(f"Pocket validation for chain {TARGET_CHAIN_ID}:")
print(f"  Requested: {len(POCKET_RESIDUE_IDS)} residues")
print(f"  Found:     {pocket_info['n_found']} residues")
print(f"  Missing:   {pocket_info['n_missing']} residues")
print(f"  Pocket sequence (one-letter): {pocket_info['pocket_sequence']}")

if pocket_info.get('warning'):
    print(f"\n  WARNING: {pocket_info['warning']}")

if pocket_info['found']:
    print("\n  Found residues:")
    for r in pocket_info['found']:
        print(f"    Residue {r['res_id']:4d}: {r['resname']}")

print()
if pocket_info['n_missing'] > 0:
    print("ACTION NEEDED: Some pocket residues were not found.")
    print("Update POCKET_RESIDUE_IDS in the config cell.")
else:
    print("All pocket residues found. Proceed to Step 4.")

## Step 3b — Visualise the Pocket (Optional)

This cell uses **py3Dmol** to render the protein chain in your browser with the
pocket residues highlighted in red.

This step is optional and non-blocking — if py3Dmol is not available (some Colab
configurations) the rest of the notebook is unaffected.

The 3D view is a quick sanity check:
- Is the highlighted region on a surface-exposed loop? (Good.)
- Is it buried in the hydrophobic core? (Suspicious — reconsider your pocket IDs.)
- Does the pocket region match the interface you intended?

In [ ]:
try:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "py3Dmol"])
    import py3Dmol

    pdb_save_path = WORKSPACE_DIR / f"{PDB_ID}.pdb"
    if not pdb_save_path.exists():
        print("PDB file not found. Run Step 1 first.")
    else:
        pdb_text_local = pdb_save_path.read_text()

        view = py3Dmol.view(width=700, height=450)
        view.addModel(pdb_text_local, "pdb")

        # Show all chains as thin cartoon, grey
        view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.6}})

        # Target chain — slightly more visible
        view.setStyle(
            {"chain": TARGET_CHAIN_ID},
            {"cartoon": {"color": "steelblue", "opacity": 0.85}},
        )

        # Pocket residues — bright red sticks + spheres
        for res_id in POCKET_RESIDUE_IDS:
            view.addStyle(
                {"chain": TARGET_CHAIN_ID, "resi": res_id},
                {"stick": {"colorscheme": "redCarbon", "radius": 0.3}},
            )
            view.addStyle(
                {"chain": TARGET_CHAIN_ID, "resi": res_id},
                {"sphere": {"color": "red", "opacity": 0.4, "radius": 0.8}},
            )

        view.zoomTo({"chain": TARGET_CHAIN_ID})
        view.show()
        print(f"Showing chain {TARGET_CHAIN_ID} (blue). "
              f"Pocket residues {POCKET_RESIDUE_IDS[:5]}{'...' if len(POCKET_RESIDUE_IDS) > 5 else ''} in red.")
        print("Check: are the red residues surface-exposed and in the region you intended?")

except ImportError:
    print("py3Dmol not available in this environment — skipping visualisation.")
    print("This is non-blocking. Continue to Step 4.")
except Exception as e:
    print(f"Visualisation failed: {e}")
    print("This is non-blocking. Continue to Step 4.")

## Step 4 — Freeze the Target Spec

We now write `target_spec.json` — a machine-readable record of all the assumptions
frozen above. Every subsequent notebook loads this file.

Do not edit this file by hand later. If you need to change assumptions,
update the config cell above and re-run this notebook.

In [ ]:
# Hard stop: do not write a spec that is known to be incomplete.
# If pocket residues are missing, the downstream scoring lane will use a silently
# wrong pocket definition. Fix the residue IDs in the config cell first.
if pocket_info['n_missing'] > 0:
    raise RuntimeError(
        f"{pocket_info['n_missing']} pocket residue(s) not found in chain "
        f"'{TARGET_CHAIN_ID}': {pocket_info['missing']}. "
        "Fix POCKET_RESIDUE_IDS in the config cell and re-run from Step 3. "
        "target_spec.json will NOT be written until all residues are confirmed."
    )

spec = make_target_spec(
    pdb_id=PDB_ID,
    chain_id=TARGET_CHAIN_ID,
    pocket_residue_ids=POCKET_RESIDUE_IDS,
    mechanism_hypothesis=MECHANISM_HYPOTHESIS,
    pocket_description=POCKET_DESCRIPTION,
    positive_reference=POSITIVE_REFERENCE,
    positive_reference_source=POSITIVE_REFERENCE_SOURCE,
)

# Add pocket property hints for the scoring lane
spec["pocket_hints"] = {
    "pocket_net_charge_hint": POCKET_NET_CHARGE_HINT,
    "pocket_hydrophobic_hint": POCKET_HYDROPHOBIC_HINT,
    "optimal_length_min": OPTIMAL_PEPTIDE_LENGTH_MIN,
    "optimal_length_max": OPTIMAL_PEPTIDE_LENGTH_MAX,
    "note": (
        "These hints guide the heuristic scoring lane in notebooks 02-04. "
        "They are not physics-derived — adjust them based on your knowledge of the pocket."
    ),
}

spec_path = WORKSPACE_DIR / "target_spec.json"
save_target_spec(spec, spec_path)
print(f"target_spec.json saved to {spec_path}")

# Show what was written
print("\nContent:")
print(json.dumps(spec, indent=2))

In [ ]:
# Export a human-readable summary
summary = format_target_summary(spec, chain_info, pocket_info)
summary_path = WORKSPACE_DIR / "target_summary.txt"
summary_path.write_text(summary)
print(summary)
print(f"\nSaved to {summary_path}")

## Outputs

| File | Description |
|------|-------------|
| `workspace/target_spec.json` | Machine-readable frozen target spec (loaded by all later notebooks) |
| `workspace/target_summary.txt` | Human-readable summary of chain/pocket assumptions |
| `workspace/{PDB_ID}.pdb` | Downloaded PDB file |

## Interpretation

Before continuing, verify:
1. The chain you selected is the correct receptor chain (not a ligand or crystal contact)
2. All pocket residues were found (no warnings above)
3. The pocket sequence makes sense for your binding hypothesis
4. Your positive reference is a real or well-motivated benchmark peptide

If any of these are wrong, fix the config cell and re-run the notebook.
Do not proceed to notebook 02 with unresolved warnings.

## Next Notebook

→ **02_reference_panel_check.ipynb**

That notebook loads `target_spec.json` and runs a calibration panel to check
whether the scoring lane can distinguish your reference from nonsense controls.